# Classificação financeira — corpus sintético
Os 12 600 registos são inventados, usando as 70 categorias ativas existentes.
Não é necessário Docker ou SQL. Dados antigos privados não são carregados.
Reiniciar o kernel se este notebook estava aberto na versão anterior.
Instalar antes as dependências no kernel: `%pip install -e "..[test]"` se estiver
na pasta notebooks, ou `%pip install -e ".[test]"` na raiz; reiniciar depois.
Métricas sintéticas não demonstram qualidade em movimentos reais.
Guardar o notebook sem outputs antes de versionar.

In [ ]:
from pathlib import Path
import os, sys, gc
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if root is None: raise RuntimeError("Abrir o notebook dentro da pasta do projeto")
os.chdir(root)
sys.path.insert(0, str(root / "src"))
from denarius_model import ROOT
from denarius_model.config import load_config, read_json, dataset_path, assert_dataset_version
from denarius_model.diagnostics import diagnose
config = load_config()
experiment = "synthetic-001"
RUN_OLLAMA_BASELINE = False
RUN_FULL_VALIDATION = False
RUN_FINAL = False
RUN_EXPORT = False
EVAL_PER_CATEGORY = None if RUN_FULL_VALIDATION else config["dataset"]["development_eval_per_category"]
diagnose()

## Preparação determinística
O gerador usa vocabulário revisto, intervalos de valores inventados e seed fixa.
As categorias foram lidas uma vez; a geração futura usa apenas a taxonomia versionada.
O teste difícil reserva contrapartes fictícias, contextos e formatos novos. Marcas
que definem labels, como ALDI/Netflix, permanecem conhecidas. Detalhes em data/synthetic/README.md.

In [ ]:
from denarius_model.synthetic import audit_public_dataset
from denarius_model.dataset import prepare
# Regenera somente data/synthetic; não lê nem substitui os datasets reais antigos.
stats = prepare()
categories = read_json(dataset_path("processed/categories.json"))
train_rows = read_json(dataset_path("splits/train.json"))
print("Dataset:", stats["dataset_version"])
print("Splits:", stats["sizes"])
assert stats["source"] == "synthetic"
assert all(r["synthetic"] for r in train_rows)

## Exploração e distribuição
Corpus equilibrado por construção. Não reproduz frequências reais de utilização.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.Series(stats["categories"]).sort_values().plot.barh(figsize=(12, 18))
plt.tight_layout()
# Apenas seis exemplos sintéticos; nenhum movimento privado.
display(pd.DataFrame(train_rows)[["description", "amount", "direction", "category"]].head(6))

## Prompts e orçamento de tokens
Só description, amount e direction entram como features. Identificadores sintéticos servem apenas para validação dos splits.

In [ ]:
from denarius_model.prompting import prompt, example
from denarius_model.config import model_source
from transformers import AutoTokenizer
synthetic = {"description":"ALDI compras supermercado SYN-LAROMA", "amount":"37.25",
             "direction":"debito_contabilistico", "category":"Despesas Correntes / Compras no ALDI"}
print(example(synthetic, categories)["completion"])
tokenizer = AutoTokenizer.from_pretrained(model_source(config["model"], True), local_files_only=True)
lengths = [len(tokenizer.encode(example(r,categories)["prompt"], add_special_tokens=False)) +
           len(tokenizer.encode(example(r,categories)["completion"], add_special_tokens=False)) + 1
           for r in train_rows]
print("Comprimento máximo de treino:", max(lengths), "/", config["model"]["max_length"])
assert max(lengths) <= config["model"]["max_length"], "Aumentar contexto ou rever formato, sem truncar categorias"
del tokenizer

## Baselines na validação
TF-IDF aprende apenas no treino. O notebook usa por omissão dois exemplos por
categoria para comparação exploratória rápida; todos os modelos usam a mesma
amostra. RUN_FULL_VALIDATION=True usa o split completo. Testes finais não são
amostrados. O baseline Ollama é opcional e diferente do HF Base.

In [ ]:
from denarius_model.evaluate import evaluate, ml_baseline, report
from denarius_model.inference import HFClassifier, OllamaClassifier, classify
ml = ml_baseline()
ml_validation = evaluate(ml, "validation", "ml", experiment, per_category=EVAL_PER_CATEGORY)
if RUN_OLLAMA_BASELINE:
    ollama_base = OllamaClassifier(config["model"]["ollama_baseline"], native_template=True)
    try:
        ollama_validation = evaluate(ollama_base, "validation", "ollama-base", experiment, per_category=EVAL_PER_CATEGORY)
    finally:
        ollama_base.unload()
base = HFClassifier(config["model"])
base_validation = evaluate(base, "validation", "base", experiment, per_category=EVAL_PER_CATEGORY)
del base; gc.collect()
import torch
torch.cuda.empty_cache()

## LoRA / QLoRA e treino
Batch 1, acumulação, checkpointing, loss na completion e early stopping por
validation loss no split completo. Três épocas configuradas. Treino demora mais
que no antigo dataset pequeno. Reutiliza adapter válido da mesma experiência;
para testar outros hyperparameters, mudar experiment para synthetic-002.

In [ ]:
from denarius_model.train import train
manifest_path = ROOT / "outputs" / experiment / "experiment.json"
adapter_dir = ROOT / "models" / experiment / "adapter"
if manifest_path.exists() and adapter_dir.exists():
    assert_dataset_version(experiment)
    previous = read_json(manifest_path)
    assert previous["configuration"] == config, "Config alterada: usar um novo experiment"
    adapter_path = str(adapter_dir)
    print("Adapter existente reutilizado:", adapter_path)
else:
    adapter_path = train(config, experiment)
gc.collect(); torch.cuda.empty_cache()

## Curvas de treino e avaliação de validação

In [ ]:
manifest = read_json(manifest_path)
history = pd.DataFrame(manifest["metrics"])
for column in ["loss", "eval_loss"]:
    if column in history:
        history.dropna(subset=[column]).plot(x="epoch", y=column)
tuned = HFClassifier(config["model"], adapter_path)
tuned_validation = evaluate(tuned, "validation", "tuned", experiment, per_category=EVAL_PER_CATEGORY)
assert tuned_validation["evaluation_rows_hash"] == base_validation["evaluation_rows_hash"]
comparison = pd.DataFrame({"TF-IDF":ml_validation, "HF Base":base_validation,
                            "Fine-tuned":tuned_validation}).T
comparison[["sample_size", "accuracy", "macro_f1", "weighted_f1", "invalid_outputs"]]

## Matriz de confusão e métricas por categoria
As categorias são as da taxonomia existente; o suporte depende da amostra exploratória/completa.

In [ ]:
display(pd.DataFrame(tuned_validation["per_category"]).T)
fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(tuned_validation["confusion_matrix"], cmap="Blues")
fig.colorbar(im, ax=ax)
ax.set_xlabel("Índice da categoria prevista"); ax.set_ylabel("Índice da categoria real")
ax.set_title("Validação sintética — índices em confusion_labels")
plt.tight_layout()
display(pd.Series(tuned_validation["confusion_labels"], name="confusion_labels"))

## Erros e inferência manual
Exemplos abaixo são apenas sintéticos. Não representam movimentos privados.

In [ ]:
predictions = read_json(ROOT / "reports" / experiment / "tuned_validation_predictions.json")
errors = [r for r in predictions if r["row"]["category"] != r["prediction"]]
display(pd.DataFrame([{ "description":r["row"]["description"], "real":r["row"]["category"],
                       "prediction":r["prediction"]} for r in errors[:6]]))
try: print(classify(tuned, synthetic, categories))
except ValueError: print("Output inválido")

## Sensibilidade
Ablações contrafactuais de montante/direção podem justificar outra categoria; taxa de mudança não equivale automaticamente a erro. scripts/robustness.py mede a validação completa separadamente.

In [ ]:
variants = [synthetic, {**synthetic,"amount":"1200.00"},
            {**synthetic,"direction":"credito_contabilistico"},
            {**synthetic,"description":"aldi compras supermercado syn-laroma"}]
for item in variants:
    try: print(classify(tuned, item, categories))
    except ValueError: print("Output inválido")

## Teste final e teste difícil
Congelar candidato/hyperparameters antes de ativar RUN_FINAL. Não usar estes
resultados para tuning. São testes sintéticos completos, não avaliação real.

In [ ]:
if RUN_FINAL:
    for split in ["test", "hard_test"]:
        evaluate(tuned, split, "tuned", experiment, final=True)
        evaluate(ml, split, "ml", experiment, final=True)
    del tuned; gc.collect(); torch.cuda.empty_cache()
    base = HFClassifier(config["model"])
    for split in ["test", "hard_test"]:
        evaluate(base, split, "base", experiment, final=True)
    del base; gc.collect(); torch.cuda.empty_cache()
report(experiment)
print("Relatório local:", ROOT / "reports" / experiment / "evaluation.md")

## Merge e preparação de exportação
Merge CPU FP32. RUN_EXPORT não converte GGUF automaticamente. Executar os
comandos llama.cpp e ollama create do README, com versões registadas localmente.
Q4_K_M e F16 devem ser comparados no mesmo backend/schema para isolar quantização.

In [ ]:
if RUN_EXPORT:
    if "tuned" in globals():
        del tuned; gc.collect(); torch.cuda.empty_cache()
    from denarius_model.export import merge
    print(merge(config, experiment))

## Inferência Ollama e benchmark
Depois de criar denarius-movement, descomentar. raw=true reproduz o prompt completion treinado; schema limita categorias. Não inferir prontidão em dados reais.

In [ ]:
# ollama = OllamaClassifier()
# print(classify(ollama, synthetic, categories))
# evaluate(ollama, "validation", "ollama", experiment, per_category=EVAL_PER_CATEGORY)
# ollama.unload()
# report(experiment)
# Benchmark frio/quente: python scripts/benchmark_ollama.py --model denarius-movement